# Image Data Collection, Processing and Facial Recognition Model

Part 2 of Formative 2 (Group 14). Each member has three photos: neutral, smiling and surprised.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.image_features import (EXPRESSIONS, augment, build_feature_table, detect_face,
                                find_images, load_image)
from src import face_model

IMAGES_DIR = "data/images"
UNAUTHORIZED_DIR = "data/unauthorized/images"
sns.set_theme(style="whitegrid")

## 1. Load and display the images

In [ ]:
images = list(find_images(IMAGES_DIR))
members = sorted({m for m, _, _ in images})
print(f"{len(images)} images from {len(members)} members")
pd.crosstab(pd.Series([m for m, _, _ in images], name="member"),
            pd.Series([e for _, e, _ in images], name="expression"))

In [ ]:
def show(img, ax, title=""):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=10)
    ax.axis("off")

lookup = {(m, e): p for m, e, p in images}
fig, axes = plt.subplots(len(members), 3, figsize=(9, 3 * len(members)), squeeze=False)
for i, m in enumerate(members):
    for j, e in enumerate(EXPRESSIONS):
        if (m, e) in lookup:
            show(load_image(lookup[(m, e)]), axes[i, j], f"{m} - {e}")
        else:
            axes[i, j].axis("off")
plt.suptitle("Original photos", y=1.0)
plt.tight_layout()
plt.show()

## 2. Face detection

Each photo is cropped to the face using OpenCV's YuNet face detector, so the background does not affect the features.

In [ ]:
fig, axes = plt.subplots(len(members), 3, figsize=(7, 2.5 * len(members)), squeeze=False)
for i, m in enumerate(members):
    for j, e in enumerate(EXPRESSIONS):
        if (m, e) not in lookup:
            axes[i, j].axis("off")
            continue
        face, found = detect_face(load_image(lookup[(m, e)]))
        show(face, axes[i, j], f"{m} - {e}" + ("" if found else " (no face found)"))
plt.suptitle("Detected faces", y=1.0)
plt.tight_layout()
plt.show()

## 3. Augmentation

Every face crop gets 8 augmented versions: rotation (+15 and -15 degrees), horizontal flip, grayscale, brighter, darker, blur and noise.

In [ ]:
for m in members:
    path = lookup.get((m, "smiling")) or next(p for mm, _, p in images if mm == m)
    face, _ = detect_face(load_image(path))
    versions = augment(face)
    fig, axes = plt.subplots(1, len(versions), figsize=(2 * len(versions), 2.4))
    for ax, (name, img) in zip(axes, versions.items()):
        show(img, ax, name)
    plt.suptitle(f"Augmentations for {m}", y=1.05)
    plt.show()

## 4. Feature extraction

For every version of every face (resized to 128x128):

- HOG (histogram of oriented gradients) on the grayscale face, which captures face shape and edges
- HSV colour histograms (16 bins per channel)
- grayscale intensity histogram (32 bins)

In [ ]:
df = build_feature_table(IMAGES_DIR)
df.to_csv("features/image_features.csv", index=False)
feature_cols = [c for c in df.columns if c not in face_model.META_COLS]
print(f"image_features.csv: {df.shape[0]} rows, {len(feature_cols)} features")
df[face_model.META_COLS + feature_cols[:3]].head()

In [ ]:
df.groupby(["member", "augmentation"]).size().unstack()

In [ ]:
gray_cols = [c for c in feature_cols if c.startswith("hist_gray")]
originals = df[df["augmentation"] == "original"]
fig, ax = plt.subplots(figsize=(8, 4))
for m in members:
    ax.plot(originals.loc[originals["member"] == m, gray_cols].mean().values, label=m)
ax.set_xlabel("Intensity bin (dark to bright)")
ax.set_ylabel("Share of pixels")
ax.set_title("Average grayscale histogram per member (original photos)")
ax.legend()
plt.show()

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform((df[feature_cols] - df[feature_cols].mean()) / (df[feature_cols].std() + 1e-8))
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=df["member"], style=df["expression"], ax=ax)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} of variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} of variance)")
ax.set_title("Image features projected to 2D (PCA)")
plt.show()

## 5. Facial recognition model

Evaluation is leave-one-expression-out: the model is trained on two expressions (with their augmentations) and tested on the third, which it has never seen in any form. This is repeated three times, once per expression.

Three models are compared on accuracy, macro F1 and log loss.

In [ ]:
results, confidences = face_model.evaluate(df)
results.round(3)

In [ ]:
summary = results.groupby("model")[["accuracy", "f1", "log_loss"]].mean().round(3)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, metric in zip(axes, ["accuracy", "f1", "log_loss"]):
    sns.barplot(data=results, x="model", y=metric, ax=ax, errorbar=None)
    ax.set_title(metric)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
best = summary["f1"].idxmax()
print("Best model:", best)
y_true, y_pred = face_model.held_out_predictions(df, best)
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=members), display_labels=members).plot(cmap="Blues")
plt.title(f"Confusion matrix ({best}, held-out expressions)")
plt.show()

## 6. Final model and rejecting unknown faces

A classifier always picks one of the known members, so two checks are used before a face is accepted:

1. the model's confidence must be above a threshold
2. the face must be close enough (cosine distance) to that member's own training images

The distance threshold comes from the held-out photos above.

In [ ]:
distances = face_model.genuine_distances(df)
bundle = face_model.train_final(df, best, confidences, distances)
face_model.save(bundle, "models/face_model.pkl")
print(f"Saved models/face_model.pkl ({best})")
print(f"Confidence threshold: {bundle['threshold']:.3f}")
print(f"Distance threshold:   {bundle['distance_threshold']:.3f}")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(distances, bins=10)
ax.axvline(bundle["distance_threshold"], color="red", linestyle="--", label="threshold")
ax.set_xlabel("Distance to closest image of the same member")
ax.set_title("Distances for held-out member photos")
ax.legend()
plt.show()

## 7. Test: known members and an unauthorized face

In [ ]:
tests = [(m, lookup[(m, "neutral")]) for m in members if (m, "neutral") in lookup]
if os.path.isdir(UNAUTHORIZED_DIR):
    tests += [("unauthorized", os.path.join(UNAUTHORIZED_DIR, f)) for f in sorted(os.listdir(UNAUTHORIZED_DIR))
              if f.lower().endswith((".jpg", ".jpeg", ".png"))]

rows = []
for who, path in tests:
    member, conf, dist = face_model.predict_face(path, bundle)
    rows.append({"photo of": who, "file": os.path.basename(path), "predicted": member or "ACCESS DENIED",
                 "confidence": round(conf, 3), "distance": round(dist, 3)})
pd.DataFrame(rows)